In [ ]:
# Async/await syntax is used to asynchronously communicate w/ Alicat controllers
# Await tells the computer to wait for the controller to respond before proceeding to the next command
# Assume each controller has its own serial port (separate connections to the host computer)
# Data recording is set up in the initial communications, runs from when the code is executed until the duration has elapsed, future plan: have the data collection start at a trigger? or just have datetime stored, then end data collection at a trigger det. by the overall experiment?
import asyncio
from alicat.driver import FlowController
import csv
import time
from datetime import datetime


filename = 'EXPERIMENT.csv'
experiment_duration = 5000

# Mass flow controller setting function

async def mfc_settings(mfc,gas_type,mfc_flow_rate):
    await mfc.set_gas(gas_type) # define gas before setting parameters
    await mfc.set_flow_rate(mfc_flow_rate)  #sccm

# Pressure controller function: Simple operation

async def pc_settings(pc,p):
    await pc.set_gas('Air')
    await pc.set_pressure(p) #bar
    
# Pressure controller: Modulation

# Modulation 1: Ramp up to a certain pressure, then ramp down (no control over exact shape or time at pressure)

async def pc_modulate1(pc,p_lo,p_hi):
    await pc.set_gas('Air')
    await pc.set_pressure(p_lo) # will wait until Alicat returns that this p is reached
    await pc.set_pressure(p_hi)

# Modulation 2: Ramp up to a certain pressure, wait, then ramp down, then wait

async def pc_modulate2(pc,p_lo,p_hi,dwell_time,duration):
    await pc.set_gas('Air')
    end_time = time.monotonic() + duration

    async def hold(pressure):
        await pc.set_pressure(pressure)
        remaining = end_time - time.monotonic()
        await asyncio.sleep(max(0, min(dwell_time, remaining)))

    try:
        while time.monotonic() < end_time:
            await hold(p_hi)
            if time.monotonic() >= end_time:
                break
            await hold(p_lo)

    finally:
        await pc.set_pressure(p_lo)

# Modulation 2: Write equation, then follow that equation ? 




async def comms(): # this entire control will run everytime comms() is called, so ayncio.run(comms()) is kind of the "trigger"
    async with FlowController(address = "SERIAL ADDRESS FOR MFC", unit = "MFC") as mfc: # define that serial port as mfc
        print(await mfc.get())
        await mfc_settings(mfc,'O2',1.0)
    async with FlowController(address = "SERIAL ADDRESS FOR PC", unit = "PC") as pc: # define that serial port as pc
        print(await pc.get())
        await pc_settings(pc,0.5) # replace with pc_modulateX() for modulation functions

        with open(filename,'w',newline = '') as file: 
            writer = csv.writer(file) # create csv to store data
            writer.writerow(['timestamp','mfc_flow_rate','pc_pressure']) # write headings

            next_time = time.monotonic() 
            while time.monotonic() - start < experiment_duration: # collects data until the experiment duration has completed
                mfc_state, pc_state = await asyncio.gather(mfc.get(),pc.get()) # store the get() dictionary for each controller at the timestamp
                writer.writerow([
                    datetime.now().isoformat(), #timestamp
                    mfc_state['mass_flow'], #mfc flow rate
                    pc_state['pressure'] #pc pressure
                    ])
                file.flush()

                next_time += 0.05 # seconds, data collection interval
                await asyncio.sleep(max(0, next_time - time.monotonic())) # maintains the correct interval time

    

asyncio.run(comms())  # or sawait get() "if there is already an event loop"

# Dictionary of current settings will be returned to confirm communications
